# Amazon Bedrock - Intents에 따른 API 호출 예시
> 이 노트북은 SageMaker JupyterLab <i><b>conda_python3</b></i> 커널에서 테스트 되었습니다.

## 0. 환경 설정

In [1]:
# 처음 실행하는 경우, install_needed를 True로 지정하고 패키지를 설치합니다.

install_needed = False

In [2]:
import sys
import IPython

if install_needed:
    print("installing deps and restarting kernel")
    !{sys.executable} -m pip install -U pip
    !{sys.executable} -m pip install -U awscli
    !{sys.executable} -m pip install -U botocore
    !{sys.executable} -m pip install -U boto3
    !{sys.executable} -m pip install -U sagemaker 
    !{sys.executable} -m pip install -U langchain
    !{sys.executable} -m pip install -U anthropic
    !{sys.executable} -m pip install -U termcolor
    !{sys.executable} -m pip install -U opensearch-py
    !{sys.executable} -m pip install -U flask-sqlalchemy
    !{sys.executable} -m pip install -U ipython
    !{sys.executable} -m pip install -U ipywidgets
    !{sys.executable} -m pip install -U jq

## 1. Bedrock 모델 정의

#### 1-1. Claude 3 Haiku 모델 정의

In [9]:
import os
from langchain_community.chat_models import BedrockChat

# LLM 정의
def get_text_response(input_content):
    llm = BedrockChat(
        credentials_profile_name=os.environ.get("BWB_PROFILE_NAME"),
        region_name="us-west-2",
        endpoint_url=os.environ.get("BWB_ENDPOINT_URL"),
        model_id="anthropic.claude-3-haiku-20240307-v1:0", # Claude 3 Haiku 
        model_kwargs={
            "max_tokens": 4096,
            "temperature": 0,
            "top_p": 0.01,
            "top_k": 0,
        }
    )
    return llm.predict(input_content)

### 1-2. Claude 3 Haiku 테스트

In [8]:
if __name__ == "__main__":
    input_text = input("Input text: ")
    response_content = get_text_response(input_content=input_text)
    print(response_content)

Input text:  당신의 이름은 무엇입니까?


안녕하세요. 제 이름은 Claude입니다. 저는 Anthropic에서 만든 AI 어시스턴트입니다. 어떤 도움이 필요하신가요?


## 2. LangChain.agents를 활용한 intent 구분 테스트

#### Intent 함수들

In [53]:
# intent 별 함수 정의
def gourmet(tool_input=None):
    description = "[현재 위치 기반 맛집 정보는 다음과 같습니다. ~~~~~]"
    return description

def thingstodo(tool_input=None):
    description = "[현재 위치 기반 즐길거리 정보는 다음과 같습니다. ~~~~~]"
    return description

def hyundai(tool_input=None):
    description = "[현대자동차는 어쩌고 저쩌고 ~~~~~]"
    return description

def ionic(tool_input=None):
    description = "[현대 아이오닉은 전기차 입니다. ~~~~~]"
    return description

def kona(tool_input=None):
    description = "[코나는 여기 전시되어 있어요. ~~~~~]"
    return description

#### intent에 대한 LLM 응답 테스트

In [54]:
import re
import os
from langchain.agents import AgentType, initialize_agent, Tool
from langchain.chat_models import BedrockChat
from langchain.prompts import PromptTemplate


# intent 매핑 딕셔너리
intent_mapping = {
    "맛집": gourmet,
    "즐길거리": thingstodo,
    "현대차": hyundai,
    "아이오닉": ionic,
    "코나": kona,
}

# intent 추출 함수
def extract_intents(text):
    intents = []
    for intent, func in intent_mapping.items():
        pattern = r'\b{}\b|\b{}[가-힣]+'.format(intent, intent)
        if re.search(pattern, text, re.IGNORECASE):
            intents.append(intent)
    return intents

# LLM 정의
llm = BedrockChat(
    credentials_profile_name=os.environ.get("BWB_PROFILE_NAME"),
    region_name="us-west-2",
    endpoint_url=os.environ.get("BWB_ENDPOINT_URL"),
    model_id="anthropic.claude-3-haiku-20240307-v1:0", # Claude 3 Haiku
    model_kwargs={
        "max_tokens": 4096,
        "temperature": 0,
        "top_p": 0.01,
        "top_k": 0,
    }
)

# Tool 정의
tools = [
    Tool(name="맛집", func=lambda tool_input: gourmet(), description="맛집 정보 API Call"),
    Tool(name="즐길거리", func=lambda tool_input: thingstodo(), description="즐길거리 정보 API Call"),
    Tool(name="현대차", func=lambda tool_input: hyundai(), description="현대차 관련 통합 정보 API Call "),
    Tool(name="아이오닉", func=lambda tool_input: ionic(), description="아이오닉 관련 통합 정보 API Call"),
    Tool(name="코나", func=lambda tool_input: kona(), description="코나 관련 통합 정보 API Call"),
]

# PromptTemplate 정의
template = """
다음 문장에서 intent를 파악하고 해당 intent에 맞는 기능을 사용하세요:
{input}
"""
prompt = PromptTemplate(template=template, input_variables=["input"])

# AgentAssistant 초기화
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=False, handle_parsing_errors=True)

if __name__ == "__main__":
    user_input = input("Input text: ")
    intents = extract_intents(user_input)

    if not intents:
        agent_response = agent.run(user_input)
        print(agent_response)
    else:
        responses = []
        for intent in intents:
            func = intent_mapping[intent]
            response = func()
            responses.append(response)
        print("\n".join(responses))

Input text:  안녕하세요. 여기 맛집이 있다면서요?


[현재 위치 기반 맛집 정보는 다음과 같습니다. ~~~~~]


---

## 3. LangChain.agents를 활용한 intent 구분 챗봇 테스트

#### Intent 함수
* 실제 개발시 각 Intent 함수는 Lambda로 분리

In [55]:
# intent 별 함수 정의
def gourmet(tool_input=None):
    description = "[현재 위치 기반 맛집 정보는 다음과 같습니다. ~~~~~]"
    return description

def thingstodo(tool_input=None):
    description = "[현재 위치 기반 즐길거리 정보는 다음과 같습니다. ~~~~~]"
    return description

def hyundai(tool_input=None):
    description = "[현대자동차는 어쩌고 저쩌고 ~~~~~]"
    return description

def ionic(tool_input=None):
    description = "[현대 아이오닉은 전기차 입니다. ~~~~~]"
    return description

def kona(tool_input=None):
    description = "[코나는 여기 전시되어 있어요. ~~~~~]"
    return description

#### Intent에 따른 응답을 반환하는 Chatbot

In [ ]:
import re
import os
from langchain.agents import AgentType, initialize_agent, Tool
from langchain.chat_models import BedrockChat
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationChain

# intent 매핑 딕셔너리
intent_mapping = {
    "맛집": gourmet,
    "즐길거리": thingstodo,
    "현대차": hyundai,
    "아이오닉": ionic,
    "코나": kona,
}

# intent 추출 함수
def extract_intents(text):
    intents = []
    for intent, func in intent_mapping.items():
        pattern = r'\b{}\b|\b{}[가-힣]+'.format(intent, intent)
        if re.search(pattern, text, re.IGNORECASE):
            intents.append(intent)
    return intents

# LLM 정의
def get_llm():
    model_kwargs = {
        "max_tokens": 1024,
        "temperature": 0,
        "top_p": 0.5,
        "top_k": 0,
        "stop_sequences": ["\n\nHuman:"],
        "messages": [{"role": "system", "content": "You are a helpful assistant."}]
    }

    llm = BedrockChat(
        credentials_profile_name=os.environ.get("BWB_PROFILE_NAME"),
        region_name="us-west-2",
        endpoint_url=os.environ.get("BWB_ENDPOINT_URL"),
        model_id="anthropic.claude-3-haiku-20240307-v1:0", # Claude 3 Haiku
        model_kwargs=model_kwargs)

    return llm

# Tool 정의
tools = [
    Tool(name="맛집", func=lambda tool_input: gourmet(), description="맛집 정보 API Call"),
    Tool(name="즐길거리", func=lambda tool_input: thingstodo(), description="즐길거리 정보 API Call"),
    Tool(name="현대차", func=lambda tool_input: hyundai(), description="현대차 관련 통합 정보 API Call "),
    Tool(name="아이오닉", func=lambda tool_input: ionic(), description="아이오닉 관련 통합 정보 API Call"),
    Tool(name="코나", func=lambda tool_input: kona(), description="코나 관련 통합 정보 API Call"),
]

# PromptTemplate 정의
template = """
다음 문장에서 intent를 파악하고 해당 intent에 맞는 기능을 사용하세요:
{input}
"""
prompt = PromptTemplate(template=template, input_variables=["input"])

# AgentAssistant 초기화
agent = initialize_agent(tools, get_llm(), agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=False, handle_parsing_errors=True)

def get_memory():
    llm = get_llm()
    chat_memory = ConversationBufferWindowMemory(human_prefix='Human', ai_prefix='Assistant')
    conversation = ConversationChain(llm=llm, verbose=False, memory=chat_memory)
    return chat_memory

def get_chat_response(input_text, memory):
    llm = get_llm()
    conversation_with_summary = ConversationChain(llm=llm, memory=memory, verbose=False)
    chat_response = conversation_with_summary.predict(input=input_text)
    return chat_response

if __name__ == "__main__":
    memory = get_memory()
    chat_history = []

    while True:
        input_text = input("Human: ")
        #print("Human:", input_text)
        chat_history.append({"role": "user", "text": input_text})

        intents = extract_intents(input_text)
        if not intents:
            chat_response = get_chat_response(input_text=input_text, memory=memory)
            print("Assistant:", chat_response)
            chat_history.append({"role": "assistant", "text": chat_response})
        else:
            responses = []
            for intent in intents:
                func = intent_mapping[intent]
                response = func()
                responses.append(response)
            print("\n".join(responses))

Human:  안녕하세요


Assistant: 안녕하세요! 저는 AI 어시스턴트입니다. 어떤 도움이 필요하신가요? 저는 다양한 주제에 대해 대화할 수 있으며, 제가 알고 있는 정보를 최대한 자세히 알려드리겠습니다. 모르는 부분이 있다면 솔직히 말씀드리겠습니다.


Human:  여기 현대차 부스죠?


[현대자동차는 어쩌고 저쩌고 ~~~~~]


Human:  코나에 대해서 알려주시고, 여기 근처 맛집도 알려줘요


[현재 위치 기반 맛집 정보는 다음과 같습니다. ~~~~~]
[코나는 여기 전시되어 있어요. ~~~~~]


In [ ]:
"""
Human:  안녕하세요

Assistant: 안녕하세요! 저는 AI 어시스턴트입니다. 어떤 도움이 필요하신가요? 저는 다양한 주제에 대해 대화할 수 있으며, 제가 알고 있는 정보를 최대한 자세히 알려드리겠습니다. 모르는 부분이 있다면 솔직히 말씀드리겠습니다.

Human:  여기 현대차 부스죠?

[현대자동차는 어쩌고 저쩌고 ~~~~~]

Human:  코나에 대해서 알려주시고, 여기 근처 맛집도 알려줘요

[현재 위치 기반 맛집 정보는 다음과 같습니다. ~~~~~]
[코나는 여기 전시되어 있어요. ~~~~~]


"""